# Instalasi Library

In [1]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42) # Call this immediately
print("Random seed set to 42 for reproducibility.")

Random seed set to 42 for reproducibility.


In [2]:
!pip install transformers torch scikit-learn pandas nltk openpyxl tqdm

import pandas as pd
import numpy as np
import glob, os
from tqdm import tqdm
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Download resource NLTK
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

2026-02-03 13:19:52.536513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770124792.733717      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770124792.791146      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

True

# Preprocess Data (Voting and Notes)

In [3]:
data_dir = '/kaggle/input/fragrantica-perfumes/'
files = [
    data_dir + "perfume-convert-split-1.xlsx",
    data_dir + "perfume-convert-split-2.xlsx",
    data_dir + "perfume-convert-split-3.xlsx",
    data_dir + "perfume-convert-split-4.xlsx",
    data_dir + "perfume-convert-split-5.xlsx"
]
print(f"Ditemukan {len(files)} file Excel (FULL DATA).")

all_cols = set()
dfs = []
for f in files:
    temp = pd.read_excel(f, nrows=1)
    all_cols.update(temp.columns)

for f in files:
    df_part = pd.read_excel(f)
    missing = list(all_cols - set(df_part.columns))
    for col in missing: df_part[col] = np.nan
    df_part = df_part[list(all_cols)]
    dfs.append(df_part)

# --- VARIABEL PENTING (df_raw) ---
# df_raw akan kita gunakan lagi di Sel 3 dan Sel 7.5
df_raw = pd.concat(dfs, ignore_index=True)
print(f"Total baris mentah digabung: {len(df_raw):,}")

longevity_map = {'longevity.very_weak': 1, 'longevity.weak': 2, 'longevity.moderate': 3, 'longevity.long_lasting': 4, 'longevity.eternal': 5}
sillage_map = {'sillage.intimate': 1, 'sillage.moderate': 2, 'sillage.strong': 3, 'sillage.enormous': 4}
value_map = {'value.way_overpriced': 1, 'value.overpriced': 2, 'value.ok': 3, 'value.good_value': 4, 'value.great_value': 5}

def weighted_score(row, mapping):
    total_votes = sum(row.get(c, 0) for c in mapping.keys() if c in row)
    if total_votes == 0: return np.nan
    score = sum(row.get(c, 0) * mapping[c] for c in mapping.keys() if c in row) / total_votes
    return score

tqdm.pandas()
group_cols = ['id', 'perfume_name', 'dizajner']
vote_cols = list(longevity_map.keys()) + list(sillage_map.keys()) + list(value_map.keys())
vote_cols = [c for c in vote_cols if c in df_raw.columns]
note_cols = [c for c in df_raw.columns if 'notes.' in c.lower()]
review_cols = [c for c in df_raw.columns if 'comment' in c.lower() or 'content' in c.lower()]

agg_dict = {c: "sum" for c in vote_cols}
for c in review_cols: agg_dict[c] = lambda x: " ||| ".join(x.dropna().astype(str))
for c in note_cols: agg_dict[c] = lambda x: "; ".join(sorted(set(x.dropna().astype(str))))

df_raw_cleaned = df_raw.dropna(subset=group_cols)
# --- VARIABEL PENTING 1 (perfumes_df) ---
perfumes_df = df_raw_cleaned.groupby(group_cols).agg(agg_dict).reset_index()
print(f"Berhasil diagregasi menjadi {len(perfumes_df):,} parfum unik.")

perfumes_df["vote_longevity"] = perfumes_df.progress_apply(lambda r: weighted_score(r, longevity_map), axis=1)
perfumes_df["vote_sillage"] = perfumes_df.progress_apply(lambda r: weighted_score(r, sillage_map), axis=1)
perfumes_df["vote_value"] = perfumes_df.progress_apply(lambda r: weighted_score(r, value_map), axis=1)

perfumes_df["all_notes_text"] = perfumes_df[note_cols].fillna("").agg("; ".join, axis=1)
perfumes_df["all_notes_text"] = perfumes_df["all_notes_text"].str.replace(r";{2,}", ";", regex=True).str.strip("; ")

drop_cols = note_cols + vote_cols
perfumes_df.drop(columns=drop_cols, inplace=True, errors="ignore")

Ditemukan 5 file Excel (FULL DATA).
Total baris mentah digabung: 152,874
Berhasil diagregasi menjadi 740 parfum unik.


100%|██████████| 740/740 [00:00<00:00, 34034.97it/s]


# Preprocess Data (Reviews)

In [4]:
# Kita gunakan df_raw yang sudah ada di memori dari Sel 2
cols_to_fill = ['id', 'perfume_name']
df_ffill = df_raw.copy() # Salin agar tidak mengubah df_raw asli
df_ffill[cols_to_fill] = df_ffill[cols_to_fill].ffill()
df_ffill = df_ffill.dropna(subset=['comments.content', 'id'])
df_ffill['id'] = df_ffill['id'].astype(int)

lemmatizer = WordNetLemmatizer()
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z0-9\s/]', '', text)
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Membersihkan semua teks komentar...")
df_ffill['cleaned_text'] = df_ffill['comments.content'].apply(clean_text)
df_ffill['word_count'] = df_ffill['cleaned_text'].apply(lambda x: len(x.split()))
# --- VARIABEL PENTING 2 (df_processed_reviews) ---
df_processed_reviews = df_ffill[df_ffill['word_count'] > 10].copy()
df_processed_reviews = df_processed_reviews[['id', 'perfume_name', 'cleaned_text']]

Membersihkan semua teks komentar...


In [5]:
# === SEL 3.5 (BARU: Simpan & Hapus Data Ulasan) ===
# Tujuan: Menyimpan data ulasan ke disk, lalu menghapusnya dari RAM
# agar RAM bersih untuk proses training (Sel 4, 5, 6).

print("Menyimpan df_processed_reviews (194k baris) ke disk...")
reviews_path = "/kaggle/working/reviews_to_predict.csv"
df_processed_reviews.to_csv(reviews_path, index=False)
print(f"Berhasil disimpan ke: {reviews_path}")

print("Menghapus df_processed_reviews dari RAM untuk menghemat memori...")
del df_processed_reviews
import gc
gc.collect()

print("✅ df_processed_reviews dihapus dari RAM. Siap untuk training.")

Menyimpan df_processed_reviews (194k baris) ke disk...
Berhasil disimpan ke: /kaggle/working/reviews_to_predict.csv
Menghapus df_processed_reviews dari RAM untuk menghemat memori...
✅ df_processed_reviews dihapus dari RAM. Siap untuk training.


# Preprocess Data (Gender Align)

In [6]:
# Kita gunakan df_raw yang sudah ada di memori dari Sel 2
cols_to_load = ['id', 'perfume_name', 'votes.male', 'votes.female', 'votes.unisex']
df_gender_raw = df_raw[cols_to_load].copy()

vote_cols = ['votes.male', 'votes.female', 'votes.unisex']
for col in vote_cols: df_gender_raw[col] = df_gender_raw[col].fillna(0)
df_gender_raw['total_votes'] = df_gender_raw[vote_cols].sum(axis=1)
df_gender = df_gender_raw.sort_values('total_votes', ascending=False).drop_duplicates(subset=['id'])

def determine_gender(row):
    v_male, v_female, v_unisex = row.get('votes.male', 0), row.get('votes.female', 0), row.get('votes.unisex', 0)
    total_votes = v_male + v_female + v_unisex
    if total_votes > 0:
        if v_male > v_female and v_male > v_unisex: return 'male'
        elif v_female > v_male and v_female > v_unisex: return 'female'
        else: return 'unisex'
    name = str(row.get('perfume_name', '')).lower()
    if "for men" in name and "women" not in name: return "male"
    if "for women" in name and "men" not in name: return "female"
    return 'unisex'

df_gender['gender'] = df_gender.apply(determine_gender, axis=1)
# --- VARIABEL PENTING 3 (gender_lookup_df) ---
gender_lookup_df = df_gender[['id', 'gender']]

# Train ABSA

In [7]:
import pandas as pd
import gc

aspect_keywords = {
    'longevity': ['longevity', 'lasts', 'lasting', 'duration', 'performance'],
    'sillage': ['sillage', 'projection', 'proyeksi', 'trail', 'aura'],
    'versatility': ['versatile', 'versatility', 'office', 'everyday', 'occasions'],
    'value': ['value', 'price', 'harga', 'worth', 'money', 'overpriced', 'good_value'],
    'scent_quality': ['scent', 'smell', 'aroma', 'fragrance', 'notes', 'dry-down', 'opening'],
    'overall': ['overall', 'satisfaction', 'conclusion', 'love it', 'hate it', 'my favorite']
}
sentiment_keywords = {
    'positive': ['love', 'great', 'amazing', 'excellent', 'good', 'best', 'beautiful', '10/10', '5/5'],
    'negative': ['poor', 'weak', 'bad', 'horrible', 'terrible', 'disappointed', 'hate', 'fades', '0/10', 'overpriced']
}
sentiment_map = {'negative': 0, 'neutral': 1, 'positive': 2}

labeled_data = []

# === MODIFIKASI DI SINI ===
# Cek apakah df_processed_reviews ada di RAM. Jika tidak, muat dari DISK.
if 'df_processed_reviews' in locals():
    print("Mengambil sampel dari DataFrame di memori...")
    sample_df = df_processed_reviews.copy()
else:
    print("Mengambil sampel dari file CSV di disk (reviews_to_predict.csv)...")
    try:
        # Muat data penuh sebentar saja
        full_df_temp = pd.read_csv("/kaggle/working/reviews_to_predict.csv")
        # Pastikan kolom cleaned_text dibaca sebagai string
        full_df_temp['cleaned_text'] = full_df_temp['cleaned_text'].astype(str)
        
        # Ambil sampel
        sample_df = full_df_temp.copy()
        
        # PENTING: Hapus data penuh segera setelah sampel diambil
        del full_df_temp
        gc.collect()
        print("Sampel berhasil diambil, sisa data dihapus dari RAM.")
    except FileNotFoundError:
        print("ERROR: File 'reviews_to_predict.csv' tidak ditemukan.")
        print("Pastikan Anda sudah menjalankan Sel 3.5 untuk menyimpan data.")
        raise

print("Memulai pelabelan otomatis (Termasuk Neutral)...")
labeled_data = []

for _, row in sample_df.iterrows():
    text = row['cleaned_text']
    # Pastikan text string
    if not isinstance(text, str): continue
        
    # 1. Tentukan Sentimen Dasar
    has_positive = any(key in text for key in sentiment_keywords['positive'])
    has_negative = any(key in text for key in sentiment_keywords['negative'])
    
    sentiment_label = sentiment_map['neutral'] # Default: Neutral (1)
    if has_positive and not has_negative: 
        sentiment_label = sentiment_map['positive'] # (2)
    elif has_negative and not has_positive: 
        sentiment_label = sentiment_map['negative'] # (0)
    
    # 2. Cari Aspek (TANPA membuang label netral)
    found_aspects = False
    for aspect, keywords in aspect_keywords.items():
        if any(key in text for key in keywords):
            labeled_data.append([text, aspect, sentiment_label])
            found_aspects = True
            
    # Jika tidak ada kata kunci aspek spesifik, masukkan ke 'overall'
    if not found_aspects: 
        labeled_data.append([text, 'overall', sentiment_label])

# Update DataFrame
absa_df = pd.DataFrame(labeled_data, columns=['text', 'aspect', 'sentiment_label']).drop_duplicates()
print(f"Data training baru berhasil dibuat: {len(absa_df)} baris.")
print("Distribusi Kelas:", absa_df['sentiment_label'].value_counts().to_dict())

Mengambil sampel dari file CSV di disk (reviews_to_predict.csv)...
Sampel berhasil diambil, sisa data dihapus dari RAM.
Memulai pelabelan otomatis (Termasuk Neutral)...
Data training baru berhasil dibuat: 228372 baris.
Distribusi Kelas: {2: 107772, 1: 106096, 0: 14504}


In [8]:
# Ambil 2 sampel acak dari setiap label (Positif, Netral, Negatif)
sample_display = absa_df.groupby('sentiment_label').apply(lambda x: x.sample(5)).reset_index(drop=True)

# Mapping agar label angka (0,1,2) jadi teks yang mudah dibaca
label_map = {0: 'Negative (0)', 1: 'Neutral (1)', 2: 'Positive (2)'}
sample_display['Label Name'] = sample_display['sentiment_label'].map(label_map)

# Tampilkan kolom yang penting saja
cols_to_show = ['text', 'aspect', 'Label Name']
print("--- CONTOH DATA HASIL LABELING OTOMATIS ---")
display(sample_display[cols_to_show])

--- CONTOH DATA HASIL LABELING OTOMATIS ---


/tmp/ipykernel_19/124423116.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_display = absa_df.groupby('sentiment_label').apply(lambda x: x.sample(5)).reset_index(drop=True)


,text,aspect,Label Name
0,surprisingly the more i have worn this the mor...,value,Negative (0)
1,i have no word for this like eating a fruit sa...,overall,Negative (0)
2,i wa excited to try this because of all the hy...,scent_quality,Negative (0)
3,cumin bomb i dont know how people like this or...,overall,Negative (0)
4,i am really bummed because i wanted to like th...,scent_quality,Negative (0)
5,one of the nicest scent ive ever tried with on...,sillage,Neutral (1)
6,i just bought this perfume but i didnt like it...,overall,Neutral (1)
7,i couldnt smell anything on the paper so i end...,scent_quality,Neutral (1)
8,aromatic and citrussy ha woody and herbal slih...,scent_quality,Neutral (1)
9,there is a new parfum version of this masterpi...,overall,Neutral (1)


# Model Training (BERT)

In [9]:
absa_df['model_input'] = absa_df['aspect'] + " [SEP] " + absa_df['text']
X, y = absa_df['model_input'].tolist(), absa_df['sentiment_label'].tolist()
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=(0.15/0.85), random_state=42, stratify=y_train_val)

class PerfumeAspectDataset(Dataset):
    def __init__(self, encodings, labels): self.encodings, self.labels = encodings, labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]); return item
    def __len__(self): return len(self.labels)

model_paths = {'BERT': 'bert-base-uncased'} # Hanya BERT
tokenizers = {name: AutoTokenizer.from_pretrained(path) for name, path in model_paths.items()}

def create_datasets_for_model(model_name):
    tokenizer = tokenizers[model_name]
    max_len = 128
    train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=max_len)
    val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=max_len)
    test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=max_len)
    return PerfumeAspectDataset(train_encodings, y_train), PerfumeAspectDataset(val_encodings, y_val), PerfumeAspectDataset(test_encodings, y_test)

def compute_metrics(pred):
    labels, preds = pred.label_ids, pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1, 'precision_macro': precision, 'recall_macro': recall}

all_test_results, trained_trainers = {}, {}
model_list_to_train = ['BERT'] 

for model_name in model_list_to_train:
    print(f"Memulai eksperimen untuk: {model_name}")
    train_dataset, val_dataset, test_dataset = create_datasets_for_model(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_paths[model_name], num_labels=3)
    training_args = TrainingArguments(output_dir=f'./results_{model_name}', num_train_epochs=5,
                                      per_device_train_batch_size=16, learning_rate=2e-5,
                                      logging_strategy="epoch", eval_strategy="epoch", save_strategy="epoch",
                                      load_best_model_at_end=True, metric_for_best_model="f1_macro",
                                      greater_is_better=True, report_to="none")
    trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,
                      eval_dataset=val_dataset, compute_metrics=compute_metrics)
    trainer.train()
    test_results = trainer.evaluate(test_dataset, metric_key_prefix="test")
    all_test_results[model_name] = test_results
    trained_trainers[model_name] = trainer
    print("--- HASIL EVALUASI MODEL (DARI TEST SET) ---")
    print(test_results)
    del model; torch.cuda.empty_cache()

results_df = pd.DataFrame(all_test_results).T
BEST_MODEL_NAME = results_df['test_f1_macro'].idxmax()
# --- VARIABEL PENTING 5 (best_trainer) ---
best_trainer = trained_trainers[BEST_MODEL_NAME]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Memulai eksperimen untuk: BERT


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,0.225400,0.166470,0.927400,0.919514,0.914047,0.925571
2,0.146100,0.133614,0.949556,0.942774,0.939145,0.946554
3,0.090200,0.121084,0.963277,0.957510,0.963374,0.951955
4,0.052300,0.156434,0.967714,0.964829,0.970441,0.959494
5,0.029200,0.157868,0.972209,0.968835,0.972987,0.964828


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked t

--- HASIL EVALUASI MODEL (DARI TEST SET) ---
{'test_loss': 0.15581844747066498, 'test_accuracy': 0.9726179355441382, 'test_f1_macro': 0.9714442842841503, 'test_precision_macro': 0.9749477818638915, 'test_recall_macro': 0.9680426181389882, 'test_runtime': 193.8805, 'test_samples_per_second': 176.686, 'test_steps_per_second': 11.043, 'epoch': 5.0}


# Predict Model ABSA

In [10]:
# === SEL 7 (PERBAIKAN FINAL: BACA DARI DISK + BATCHING) ===
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import torch
import gc
import os
import glob

# Pastikan model dan tokenizer sudah ada dari Sel 6
if 'BEST_MODEL_NAME' not in locals() or 'best_trainer' not in locals():
    print("ERROR: 'best_trainer' tidak ditemukan. Jalankan Sel 6.")
    raise ValueError("best_trainer not in memory")
else:
    print(f"Menggunakan model terbaik ({BEST_MODEL_NAME}) untuk memprediksi semua ulasan...")

    # Ukuran batch (1000 - 2500 aman karena kita streaming dari disk)
    BATCH_SIZE = 2500

    tokenizer = tokenizers[BEST_MODEL_NAME]
    all_aspects = list(aspect_keywords.keys())

    class PredictionDataset(Dataset):
        def __init__(self, encodings): self.encodings = encodings
        def __getitem__(self, idx): return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        def __len__(self): return len(self.encodings['input_ids'])

    # Lokasi file yang disimpan di Sel 3.5
    reviews_path = "/kaggle/working/reviews_to_predict.csv"

    if not os.path.exists(reviews_path):
        print(f"ERROR: File {reviews_path} tidak ditemukan.")
        print("Pastikan Anda sudah menjalankan Sel 3.5 untuk menyimpan data sebelum dihapus dari RAM.")
        raise FileNotFoundError(reviews_path)

    # Bersihkan file batch lama jika ada
    old_batch_files = glob.glob("/kaggle/working/batch_preds_*.csv")
    if old_batch_files:
        print(f"Membersihkan {len(old_batch_files)} file batch lama...")
        for f in old_batch_files: os.remove(f)

    print(f"Memulai proses prediksi dari file: {reviews_path}")
    print(f"Memproses dalam cicilan (chunksize) sebesar {BATCH_SIZE} ulasan...")

    # --- LOOP UTAMA (STREAMING DARI DISK) ---
    # Kita gunakan pd.read_csv dengan chunksize agar RAM tidak pernah penuh
    for i, batch_df in enumerate(pd.read_csv(reviews_path, chunksize=BATCH_SIZE)):
        
        # Pastikan kolom teks dibaca sebagai string (hindari error NaN)
        batch_df['cleaned_text'] = batch_df['cleaned_text'].astype(str)

        start_index = i * BATCH_SIZE
        print(f"--- Memproses Batch ke-{i}: Baris {start_index} s/d {start_index + len(batch_df)} ---")

        batch_reviews_list = batch_df['cleaned_text'].tolist()
        batch_review_ids = batch_df['id'].tolist()
        
        pred_texts, pred_ids, pred_aspects = [], [], []

        for j, text in enumerate(batch_reviews_list):
            for aspect in all_aspects:
                pred_texts.append(f"{aspect} [SEP] {text}")
                pred_ids.append(batch_review_ids[j])
                pred_aspects.append(aspect)

        if not pred_texts:
            print("Batch kosong, melanjutkan..."); continue

        # Tokenisasi & Prediksi
        pred_encodings = tokenizer(pred_texts, truncation=True, padding=True, max_length=128)
        pred_dataset = PredictionDataset(pred_encodings)

        # Prediksi (RAM naik sebentar di sini, tapi aman karena batch terbatas)
        raw_preds, _, _ = best_trainer.predict(pred_dataset)
        predictions = np.argmax(raw_preds, axis=1)

        sentiment_score_map = {0: -1, 1: 0, 2: 1}
        scores = [sentiment_score_map[p] for p in predictions]

        results_batch_raw = pd.DataFrame({
            'id': pred_ids,
            'aspect': pred_aspects,
            'sentiment_score': scores
        })

        # Simpan hasil batch langsung ke disk
        batch_filename = f"/kaggle/working/batch_preds_{start_index}.csv"
        results_batch_raw.to_csv(batch_filename, index=False)
        print(f"✅ Batch {i} disimpan: {batch_filename}")

        # Bersihkan memori (Wajib!)
        del batch_df, batch_reviews_list, batch_review_ids, pred_texts
        del pred_encodings, pred_dataset, raw_preds, results_batch_raw
        gc.collect()

    print("\n✅ SEL 7 SELESAI: Semua prediksi berhasil disimpan ke disk.")

Menggunakan model terbaik (BERT) untuk memprediksi semua ulasan...
Memulai proses prediksi dari file: /kaggle/working/reviews_to_predict.csv
Memproses dalam cicilan (chunksize) sebesar 2500 ulasan...
--- Memproses Batch ke-0: Baris 0 s/d 2500 ---
✅ Batch 0 disimpan: /kaggle/working/batch_preds_0.csv
--- Memproses Batch ke-1: Baris 2500 s/d 5000 ---


✅ Batch 1 disimpan: /kaggle/working/batch_preds_2500.csv
--- Memproses Batch ke-2: Baris 5000 s/d 7500 ---


✅ Batch 2 disimpan: /kaggle/working/batch_preds_5000.csv
--- Memproses Batch ke-3: Baris 7500 s/d 10000 ---


✅ Batch 3 disimpan: /kaggle/working/batch_preds_7500.csv
--- Memproses Batch ke-4: Baris 10000 s/d 12500 ---


✅ Batch 4 disimpan: /kaggle/working/batch_preds_10000.csv
--- Memproses Batch ke-5: Baris 12500 s/d 15000 ---


...

✅ Batch 46 disimpan: /kaggle/working/batch_preds_115000.csv
--- Memproses Batch ke-47: Baris 117500 s/d 120000 ---


✅ Batch 47 disimpan: /kaggle/working/batch_preds_117500.csv
--- Memproses Batch ke-48: Baris 120000 s/d 122500 ---


✅ Batch 48 disimpan: /kaggle/working/batch_preds_120000.csv
--- Memproses Batch ke-49: Baris 122500 s/d 125000 ---


✅ Batch 49 disimpan: /kaggle/working/batch_preds_122500.csv
--- Memproses Batch ke-50: Baris 125000 s/d 127500 ---


✅ Batch 50 disimpan: /kaggle/working/batch_preds_125000.csv
--- Memproses Batch ke-51: Baris 127500 s/d 130000 ---


# Agregasi ABSA dan Profil Data

In [ ]:
# === SEL 8 (PERBAIKAN STRUKTUR DATAFRAME) ===
import pandas as pd
import glob
import gc
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# ======================================================================
# BAGIAN A: AGREASI SKOR ABSA (HEMAT MEMORI)
# ======================================================================
print("\n--- SEL 8 (Bagian A): Mengagregasi batch prediksi dari disk ---")

batch_files = glob.glob("/kaggle/working/batch_preds_*.csv")
if not batch_files:
    print("ERROR: Tidak ada file 'batch_preds_*.csv' yang ditemukan. Jalankan Sel 7 terlebih dahulu.")
    raise ValueError("File batch tidak ditemukan")
else:
    print(f"Ditemukan {len(batch_files)} file batch prediksi.")
    
    agg_list = []
    for f in batch_files:
        # print(f"Memproses file batch: {f}...") # Opsional: dimatikan agar log tidak kepanjangan
        batch_df = pd.read_csv(f)
        
        # Agregasi per file
        batch_agg = batch_df.groupby(['id', 'aspect'])['sentiment_score'].agg(['sum', 'count'])
        agg_list.append(batch_agg)
        
        del batch_df, batch_agg
        gc.collect()

    # Gabungkan semua hasil agregasi
    all_agg_df = pd.concat(agg_list)
    
    # Hitung Agregasi Final (Sum semua batch)
    final_scores_df = all_agg_df.groupby(level=[0, 1]).sum() 
    
    # Hitung Rata-rata (MEAN)
    final_scores_df['mean_score'] = final_scores_df['sum'] / final_scores_df['count']
    
    # === PERBAIKAN DI SINI ===
    # 1. Pivot tabel (Index='id', Columns='aspect')
    absa_scores_pivot = final_scores_df.reset_index().pivot(index='id', columns='aspect', values='mean_score')
    
    # 2. Ganti nama kolom aspek (misal: 'longevity' -> 'absa_longevity')
    absa_scores_pivot.columns = [f"absa_{col}" for col in absa_scores_pivot.columns]
    
    # 3. PENTING: Reset Index agar 'id' kembali menjadi kolom biasa
    absa_scores_pivot = absa_scores_pivot.reset_index()
    
    # Bersihkan nama index/columns axis agar rapi
    absa_scores_pivot.columns.name = None
    
    print("Agregasi skor ABSA dari disk selesai.")
    print(f"Kolom ABSA yang terbentuk: {list(absa_scores_pivot.columns)}")

# ======================================================================
# BAGIAN B: PENGGABUNGAN DATA & REKOMENDASI
# ======================================================================
print("\n--- SEL 8 (Bagian B): Memuat data Voting, Notes, & Gender ---")

try:
    # Cek keberadaan variabel perfumes_df (dari Sel 2)
    if 'perfumes_df' not in locals():
        print("WARNING: 'perfumes_df' tidak ada di memori. Pastikan Anda menjalankan Sel 2.")
        # Jika Anda menyimpannya ke disk sebelumnya, uncomment baris bawah:
        # perfumes_df = pd.read_csv("/kaggle/working/perfumes_cleaned.csv") 
        raise ValueError("perfumes_df hilang dari memori.")
    
    # Cek keberadaan variabel gender_lookup_df (dari Sel 4)
    if 'gender_lookup_df' not in locals():
        print("WARNING: 'gender_lookup_df' tidak ada di memori. Pastikan Anda menjalankan Sel 4.")
        raise ValueError("gender_lookup_df hilang dari memori.")

    # --- MERGE DATA ---
    # Sekarang aman karena 'id' adalah kolom di kedua DataFrame
    final_profiles_temp = pd.merge(perfumes_df, absa_scores_pivot, on='id', how='left')
    final_profiles = pd.merge(final_profiles_temp, gender_lookup_df, on='id', how='left')

    # Isi nilai kosong (NaN)
    final_profiles['gender'] = final_profiles['gender'].fillna('unisex')
    final_profiles['all_notes_text'] = final_profiles['all_notes_text'].fillna("")
    
    # Isi nilai skor ABSA yang NaN dengan 0 (atau nilai netral)
    absa_cols = [col for col in final_profiles.columns if 'absa_' in col]
    final_profiles[absa_cols] = final_profiles[absa_cols].fillna(0)

    print("Data profil final (Voting, ABSA, Notes, Gender) berhasil digabung.")
    
    # Simpan file final
    output_path = "/kaggle/working/final_perfume_profiles_combined_FULL.csv"
    final_profiles.to_csv(output_path, index=False)
    print(f"Profil parfum gabungan disimpan ke: {output_path}")

except Exception as e:
    print(f"ERROR di Bagian B: {e}")
    raise(e)

# ======================================================================
# BAGIAN C: LOGIKA REKOMENDASI (FILTER & RE-RANK)
# ======================================================================
print("\n--- SEL 8 (Bagian C): Mempersiapkan fungsi rekomendasi ---")

# 1. Matriks Kemiripan Notes
print("Menghitung TF-IDF Notes...")
tfidf = TfidfVectorizer(max_features=500)
notes_matrix = tfidf.fit_transform(final_profiles['all_notes_text'])
notes_cosine_sim_matrix = cosine_similarity(notes_matrix)

# 2. Skor Performa
print("Menghitung Skor Performa...")
performance_weights = {
    "absa_scent_quality": 1.5, "absa_longevity": 1.0, "absa_sillage": 0.75,
    "absa_overall": 1.25, "absa_versatility": 0.5, "absa_value": 0.5,
    "vote_longevity": 0.5, "vote_sillage": 0.3, "vote_value": 0.25
}
feature_cols = [col for col in performance_weights.keys() if col in final_profiles.columns]
features_df = final_profiles[feature_cols].fillna(0) 

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_df)
features_scaled_df = pd.DataFrame(features_scaled, columns=feature_cols)

final_profiles['performance_score'] = 0
for col in feature_cols:
    final_profiles['performance_score'] += features_scaled_df[col] * performance_weights[col]

scaler_final = MinMaxScaler()
final_profiles['performance_score_norm'] = scaler_final.fit_transform(final_profiles[['performance_score']])

# Lookup Index agar cepat
indices = pd.Series(final_profiles.index, index=final_profiles['id']).drop_duplicates()

def get_recommendations_filter_rerank(perfume_id, top_k=5, candidate_pool=50):
    try:
        if perfume_id not in indices:
            return f"ID {perfume_id} tidak ditemukan dalam database."
            
        idx = indices[perfume_id]
        
        # Ambil data input
        input_perfume_data = final_profiles.iloc[idx]
        perfume_name = input_perfume_data['perfume_name']
        input_gender = input_perfume_data['gender']
        
        print(f"--- Rekomendasi untuk: {perfume_name} (Gender: {input_gender}) ---")
        
        # Tentukan target gender
        if input_gender == 'male': target_genders = ['male', 'unisex']
        elif input_gender == 'female': target_genders = ['female', 'unisex']
        else: target_genders = ['male', 'female', 'unisex']

        # Ambil Similarity Score (Notes)
        note_sim_scores = list(enumerate(notes_cosine_sim_matrix[idx]))
        note_sim_scores = sorted(note_sim_scores, key=lambda x: x[1], reverse=True)
        
        # Ambil Top-N Kandidat berdasarkan Notes
        candidate_indices = [i[0] for i in note_sim_scores[1:candidate_pool+1]]
        candidates_df = final_profiles.iloc[candidate_indices].copy()
        
        # Filter Gender
        candidates_filtered = candidates_df[candidates_df['gender'].isin(target_genders)]
        
        if candidates_filtered.empty:
            return "Tidak ada rekomendasi yang cocok setelah filter gender."
            
        # Re-rank berdasarkan Performance Score (Voting + ABSA)
        final_recommendations = candidates_filtered.sort_values('performance_score_norm', ascending=False)
        
        cols_show = ['id', 'perfume_name', 'dizajner', 'gender', 'performance_score_norm']
        return final_recommendations[cols_show].head(top_k)
        
    except Exception as e:
        return f"Error di fungsi rekomendasi: {e}"

print("✅ Sel 8 Selesai: Sistem Siap!")

# Uji Coba
print("\n--- TEST RUN (ID 17) ---")
res = get_recommendations_filter_rerank(17)
display(res)

# Rekomendasi Model

In [ ]:
# Gunakan final_profiles (dari Sel 8)
tfidf = TfidfVectorizer(max_features=500)
notes_matrix = tfidf.fit_transform(final_profiles['all_notes_text'])
notes_cosine_sim_matrix = cosine_similarity(notes_matrix)
print("Matriks Kemiripan Aroma selesai dibuat.")

performance_weights = {
    "absa_scent_quality": 1.5, "absa_longevity": 1.0, "absa_sillage": 0.75,
    "absa_overall": 1.25, "absa_versatility": 0.5, "absa_value": 0.5,
    "vote_longevity": 0.5, "vote_sillage": 0.3, "vote_value": 0.25
}
feature_cols = [col for col in performance_weights.keys() if col in final_profiles.columns]
features_df = final_profiles[feature_cols].fillna(0) 
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_df)
features_scaled_df = pd.DataFrame(features_scaled, columns=feature_cols)
final_profiles['performance_score'] = 0
for col in feature_cols:
    final_profiles['performance_score'] += features_scaled_df[col] * performance_weights[col]
scaler_final = MinMaxScaler()
final_profiles['performance_score_norm'] = scaler_final.fit_transform(final_profiles[['performance_score']])
print("Skor Performa (penguat) selesai dihitung.")

indices = pd.Series(final_profiles.index, index=final_profiles['id']).drop_duplicates()

def get_recommendations_filter_rerank(perfume_id, top_k=5, candidate_pool=50):
    try:
        input_perfume_data = final_profiles.loc[final_profiles['id'] == perfume_id].iloc[0]
        perfume_name, input_gender = input_perfume_data['perfume_name'], input_perfume_data['gender']
        print(f"--- Rekomendasi (Filter & Re-rank) untuk: {perfume_name} (Gender: {input_gender}) ---")
        
        if input_gender == 'male': target_genders = ['male', 'unisex']
        elif input_gender == 'female': target_genders = ['female', 'unisex']
        else: target_genders = ['male', 'female', 'unisex']
        print(f"Target Gender Rekomendasi: {target_genders}")

        idx = indices[perfume_id]
        note_sim_scores = list(enumerate(notes_cosine_sim_matrix[idx]))
        note_sim_scores = sorted(note_sim_scores, key=lambda x: x[1], reverse=True)
        candidate_indices = [i[0] for i in note_sim_scores[1:candidate_pool+1]]
        candidates_df = final_profiles.iloc[candidate_indices][['id', 'perfume_name', 'performance_score_norm', 'gender']]
        
        print(f"Tahap 1: Ditemukan {len(candidates_df)} kandidat aroma.")
        candidates_filtered_gender = candidates_df[candidates_df['gender'].isin(target_genders)]
        print(f"Tahap 1.5: {len(candidates_filtered_gender)} kandidat lolos filter gender.")
        
        if candidates_filtered_gender.empty:
            return "Tidak ditemukan parfum dengan aroma serupa DAN gender yang sesuai."
        final_recommendations = candidates_filtered_gender.sort_values('performance_score_norm', ascending=False)
        return final_recommendations.head(top_k)
    except Exception as e:
        return f"Error (Filter & Rerank): {e}"

# Rekomendasi berdasarkan 'REMINDS'

In [ ]:
# === SEL 12 (REVISI: PARSING REMINDS YANG BENAR) ===
import re
import pandas as pd

# Kita gunakan df_raw (dari Sel 2)
# Pastikan kita mengambil data reminds yang tidak kosong
reminds_lookup_df = df_raw[['id', 'perfume_name', 'reminds']].copy()

# Bersihkan data: Hapus yang ID atau Reminds-nya kosong
perfume_lookup = reminds_lookup_df.dropna(subset=['id', 'perfume_name', 'reminds']).drop_duplicates(subset=['id'])
perfume_lookup['id'] = perfume_lookup['id'].astype(int)

# Buat Dictionary Lookup
# Kita pakai lowercase untuk key nama agar pencocokan lebih mudah
name_to_id_map = {name.lower().strip(): pid for name, pid in zip(perfume_lookup['perfume_name'], perfume_lookup['id'])}
id_to_name_map = perfume_lookup.set_index('id')['perfume_name'].to_dict()
id_to_reminds_map = perfume_lookup.set_index('id')['reminds'].to_dict()

print(f"Lookup table 'reminds' berhasil dibuat untuk {len(id_to_reminds_map)} parfum.")

def get_recommendations_by_similar(perfume_id, top_k=5):
    try:
        # Ambil nama parfum input
        if perfume_id not in id_to_name_map:
            return pd.DataFrame() # Return kosong jika ID tidak ada
            
        perfume_name = id_to_name_map[perfume_id]
        print(f"--- Rekomendasi (Similar Items) untuk: {perfume_name} ---")
        
        reminds_string = id_to_reminds_map.get(perfume_id)
        if not reminds_string:
            return "Tidak ditemukan data 'similar perfumes' (reminds) untuk parfum ini."
            
        # --- LOGIKA PARSING BARU (ROBUST) ---
        # 1. Pecah string berdasarkan tanda pipa '|'
        raw_segments = str(reminds_string).split('|')
        
        recommendations = []
        for segment in raw_segments:
            # 2. Ambil nama sebelum tanda kurung buka '('
            # Contoh: "Parfum A (👍10)" -> "Parfum A "
            clean_name = segment.split('(')[0].strip()
            
            # 3. Cari ID-nya di map (pakai lowercase)
            clean_name_lower = clean_name.lower()
            
            if clean_name_lower in name_to_id_map:
                similar_id = name_to_id_map[clean_name_lower]
                # Jangan merekomendasikan diri sendiri
                if similar_id != perfume_id:
                    recommendations.append({'id': similar_id, 'perfume_name': clean_name})
        
        # Buat DataFrame hasil
        if not recommendations:
            return pd.DataFrame()
            
        return pd.DataFrame(recommendations).head(top_k)
        
    except Exception as e:
        print(f"Error (Similar): {e}")
        return pd.DataFrame()

# Test fungsi revisi
print("\n--- TEST FUNGSI REVISI (ID 17) ---")
display(get_recommendations_by_similar(17))

# Rekomendasi Hybrid (Final Testing)

In [ ]:
def get_hybrid_recommendations_final(perfume_id, top_k=5, candidate_pool=50):
    print(f"\n--- MEMPROSES REKOMENDASI HIBRIDA FINAL UNTUK ID: {perfume_id} ---")
    
    # 1. Dapatkan Rekomendasi dari Sel 9 (Filter & Rerank)
    recs_profile = get_recommendations_filter_rerank(perfume_id, top_k=top_k, candidate_pool=candidate_pool)
    
    # 2. Dapatkan Rekomendasi dari Sel 10 (Similar Items)
    recs_similar = get_recommendations_by_similar(perfume_id, top_k=top_k)
    
    # 3. Gabungkan Hasilnya
    if isinstance(recs_profile, pd.DataFrame) and isinstance(recs_similar, pd.DataFrame):
        hybrid_recs_df = pd.concat([recs_profile, recs_similar], ignore_index=True)
        hybrid_recs_df = hybrid_recs_df.drop_duplicates(subset=['id']).reset_index(drop=True)
        final_hybrid_recs = hybrid_recs_df.head(top_k)
    elif isinstance(recs_profile, pd.DataFrame):
        final_hybrid_recs = recs_profile
    elif isinstance(recs_similar, pd.DataFrame):
        final_hybrid_recs = recs_similar
    else:
        return "Tidak dapat menghasilkan rekomendasi dari kedua sistem."

    print("\n--- REKOMENDASI HIBRIDA FINAL ---")
    return final_hybrid_recs

print("✅ Sel 11 Selesai: Fungsi `get_hybrid_recommendations_final` siap.")

# ==============================================================================
# FINAL AUTOMATED TEST RUN (MULTI-SCENARIO)
# Kode ini akan berjalan otomatis saat Save & Commit untuk 3 skenario berbeda
# ==============================================================================

# Daftar skenario pengujian: [ID, Nama Parfum, Kategori]
test_scenarios = [
    {"id": 17,  "name": "Terre d'Hermès",      "category": "MALE (Woody Spicy)"},
    {"id": 611, "name": "Coco Mademoiselle",   "category": "FEMALE (Amber Floral)"},
    {"id": 276, "name": "CK One",              "category": "UNISEX (Citrus Aromatic)"}
]

print("\n" + "#"*60)
print("   MEMULAI PENGUJIAN OTOMATIS (MALE, FEMALE, UNISEX)   ")
print("#"*60)

for scenario in test_scenarios:
    p_id = scenario["id"]
    p_name = scenario["name"]
    p_cat = scenario["category"]
    
    print(f"\n\n>>> 🧪 SKENARIO {p_cat}")
    print(f">>> Input Parfum: {p_name} (ID: {p_id})")
    print("-" * 50)
    
    try:
        # Memanggil fungsi rekomendasi final
        # Pastikan fungsi 'get_hybrid_recommendations_final' sudah didefinisikan di cell sebelumnya
        results = get_hybrid_recommendations_final(p_id, top_k=5)
        
        # Menampilkan hasil dataframe
        display(results)
        
    except Exception as e:
        print(f"❌ Error pada pengujian {p_name}: {e}")

print("\n" + "="*60)
print("✅ SEMUA PENGUJIAN SELESAI.")
print("="*60)

# Hit Rate Rekomendasi (Reminds vs Predicted/Aggregated)

In [ ]:
import random
import numpy as np
from tqdm import tqdm

# --- FUNGSI EVALUASI ROBUSTNESS (PENGGANTI HIT RATE) ---
def evaluate_robustness(sample_size=200, top_k=5):
    # Ambil semua ID yang ada di sistem
    all_ids = list(name_to_id_map.values())
    
    # Ambil sampel acak
    actual_sample_size = min(sample_size, len(all_ids))
    test_samples = random.sample(all_ids, actual_sample_size)
    
    print(f"Melakukan evaluasi robustness pada {actual_sample_size} parfum...")
    
    gender_hits = 0
    total_recs_count = 0
    unique_brands_accumulated = 0
    
    # Buat lookup dictionary untuk gender & brand agar cepat
    # Asumsi 'final_profiles' ada di memori (dari Sel 8)
    if 'gender' in final_profiles.columns and 'dizajner' in final_profiles.columns:
        id_to_gender = final_profiles.set_index('id')['gender'].to_dict()
        id_to_brand = final_profiles.set_index('id')['dizajner'].to_dict()
    else:
        print("[ERROR] Kolom 'gender' atau 'dizajner' tidak ditemukan di final_profiles")
        return 0, 0
        
    for pid in tqdm(test_samples):
        if pid not in id_to_gender: continue # Skip jika ID error
            
        input_gender = id_to_gender[pid]
        input_brand = id_to_brand[pid]
        
        # Logika Gender yang Benar (Sesuai Tesis)
        if input_gender == 'male': 
            valid_genders = ['male', 'unisex']
        elif input_gender == 'female': 
            valid_genders = ['female', 'unisex']
        else: 
            valid_genders = ['male', 'female', 'unisex']
            
        try:
            # Ambil Rekomendasi (Hanya Komponen Filter & Rerank)
            recs_df = get_recommendations_filter_rerank(pid, top_k=top_k, candidate_pool=50)
            
            if isinstance(recs_df, str) or recs_df.empty:
                continue
                
            # 1. Hitung Gender Consistency
            # Berapa banyak hasil rekomendasi yang gendernya sesuai aturan?
            for _, row in recs_df.iterrows():
                rec_gender = row['gender']
                if rec_gender in valid_genders:
                    gender_hits += 1
                total_recs_count += 1
            
            # 2. Hitung Brand Diversity (Serendipity)
            # Apakah sistem merekomendasikan brand selain brand input?
            rec_brands = set(recs_df['dizajner'].tolist())
            # Jika ada minimal 1 brand yang BEDA dari input, kita anggap diverse
            if len(rec_brands) > 1 or (len(rec_brands)==1 and input_brand not in rec_brands):
                unique_brands_accumulated += 1
                
        except Exception as e:
            continue

    # Hitung Skor Rata-rata
    if total_recs_count > 0:
        gender_consistency_score = (gender_hits / total_recs_count) * 100
        diversity_score = (unique_brands_accumulated / actual_sample_size) * 100
    else:
        gender_consistency_score = 0
        diversity_score = 0
        
    return gender_consistency_score, diversity_score

# --- JALANKAN ---
try:
    score_gender, score_div = evaluate_robustness(200, 5)

    print("\n" + "="*50)
    print("HASIL UNTUK DI-COPY KE TESIS (BAB 4.3.3)")
    print("="*50)
    print(f"Gender Consistency Score : {score_gender:.2f}%")
    print(f"Brand Discovery Rate     : {score_div:.2f}%")
    print("="*50)
except Exception as e:
    print(f"Error: {e}")
    print("Pastikan Sel 8 (Final Profiles) sudah dijalankan!")

# Visualisasi

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from torch.utils.data import Dataset
from transformers import AutoTokenizer

# --- 1. SETUP ULANG (Hanya memuat ulang data, tidak training ulang) ---
class PerfumeAspectDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

print("Memuat ulang data uji (Test Set) agar Confusion Matrix bisa dibuat...")

# Muat data mentah & ambil sampel yang SAMA
if 'df_processed_reviews' not in locals():
    try:
        temp_df = pd.read_csv("/kaggle/working/reviews_to_predict.csv")
        temp_df['cleaned_text'] = temp_df['cleaned_text'].astype(str)
        sample_df = temp_df.copy()
    except:
        print("Error: File data tidak ditemukan. Pastikan Anda menjalankan notebook secara urut.")
        sample_df = pd.DataFrame() # Fallback empty
else:
    sample_df = df_processed_reviews.copy()

# --- 2. LABELING ULANG (LOGIKA BARU: TERMASUK NETRAL) ---
# Kita samakan logikanya dengan Cell 6 yang baru
aspect_keywords = {
    'longevity': ['longevity', 'lasts', 'lasting', 'duration', 'performance'],
    'sillage': ['sillage', 'projection', 'proyeksi', 'trail', 'aura'],
    'versatility': ['versatile', 'versatility', 'office', 'everyday', 'occasions'],
    'value': ['value', 'price', 'harga', 'worth', 'money', 'overpriced', 'good_value'],
    'scent_quality': ['scent', 'smell', 'aroma', 'fragrance', 'notes', 'dry-down', 'opening'],
    'overall': ['overall', 'satisfaction', 'conclusion', 'love it', 'hate it', 'my favorite']
}
sentiment_keywords = {
    'positive': ['love', 'great', 'amazing', 'excellent', 'good', 'best', 'beautiful', '10/10', '5/5'],
    'negative': ['poor', 'weak', 'bad', 'horrible', 'terrible', 'disappointed', 'hate', 'fades', '0/10', 'overpriced']
}
sentiment_map = {'negative': 0, 'neutral': 1, 'positive': 2}

labeled_data = []
print("Melabel ulang data sampel (Termasuk Neutral)...")

for _, row in sample_df.iterrows():
    text = row['cleaned_text']
    if not isinstance(text, str): continue
    
    # 1. Tentukan Sentimen
    has_positive = any(key in text for key in sentiment_keywords['positive'])
    has_negative = any(key in text for key in sentiment_keywords['negative'])
    
    sentiment_label = sentiment_map['neutral'] # Default 1
    if has_positive and not has_negative: 
        sentiment_label = sentiment_map['positive'] # 2
    elif has_negative and not has_positive: 
        sentiment_label = sentiment_map['negative'] # 0
    
    # 2. Cari Aspek (TANPA filter membuang netral)
    found_aspects = False
    for aspect, keywords in aspect_keywords.items():
        if any(key in text for key in keywords):
            labeled_data.append([text, aspect, sentiment_label])
            found_aspects = True
            
    if not found_aspects: 
        labeled_data.append([text, 'overall', sentiment_label])

absa_df_recreated = pd.DataFrame(labeled_data, columns=['text', 'aspect', 'sentiment_label']).drop_duplicates()

# --- 3. PROSES DATASET UNTUK PREDIKSI ---
# Tokenisasi Ulang
separator = " </s> " if "roberta" in BEST_MODEL_NAME else " [SEP] "
absa_df_recreated['model_input'] = absa_df_recreated['aspect'] + separator + absa_df_recreated['text']

X_re = absa_df_recreated['model_input'].tolist()
y_re = absa_df_recreated['sentiment_label'].tolist()

# Split Data (Harus sama random_state-nya dengan Cell 7)
X_train_val_re, X_test_re, y_train_val_re, y_test_re = train_test_split(
    X_re, y_re, test_size=0.15, random_state=42, stratify=y_re
)

tokenizer = AutoTokenizer.from_pretrained(model_paths.get(BEST_MODEL_NAME, 'distilbert-base-uncased'))
test_encodings_re = tokenizer(X_test_re, truncation=True, padding=True, max_length=128)
test_dataset = PerfumeAspectDataset(test_encodings_re, y_test_re)

# --- 4. BUAT GAMBAR MATRIKS ---
print("Membuat Prediksi & Confusion Matrix...")
raw_pred, labels, _ = best_trainer.predict(test_dataset)
y_pred = np.argmax(raw_pred, axis=1)
y_true = labels

# Paksa label 0, 1, 2 agar matriks selalu 3x3
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
class_names = ['Negative', 'Neutral', 'Positive']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 14})

plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.title(f'Confusion Matrix - {BEST_MODEL_NAME}', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(f'confusion_matrix_{BEST_MODEL_NAME}_final.png', dpi=300)
plt.show()

print(f"✅ Selesai. Gambar disimpan sebagai: confusion_matrix_{BEST_MODEL_NAME}_final.png")
# Cek Diagnostik
unique_true, counts_true = np.unique(y_true, return_counts=True)
print("\n--- DATA DIAGNOSTIC (Harus ada 0, 1, 2) ---")
print("True Labels present:   ", dict(zip(unique_true, counts_true)))

In [ ]:
# ============================================================
# FINAL EVALUATION: EFFICIENCY + ABLATION (NOTEBOOK-ALIGNED)
# ============================================================

import time
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# ------------------------------------------------------------
# EXPLICIT VARIABLES (MATCHING YOUR NOTEBOOK)
# ------------------------------------------------------------

target_df = df_processed_reviews          # ✅ exists
notes_col = "text"                        # ✅ review text
name_col  = "perfume_name"                # ✅ correct column name
target_model = model                      # ✅ fine-tuned ABSA model
target_tokenizer = tokenizer              # ✅ tokenizer

print("Using explicit notebook variables ✔")

# ============================================================
# EFFICIENCY METRICS (Reviewer 3)
# ============================================================

print("\n[Metric] Measuring Model Efficiency...")

target_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_model.to(device)

# ---- Model size ----
param_size = sum(p.numel() * p.element_size() for p in target_model.parameters())
buffer_size = sum(b.numel() * b.element_size() for b in target_model.buffers())
size_mb = (param_size + buffer_size) / (1024 ** 2)

# ---- Inference time ----
dummy_text = "The fragrance opens with citrus notes and a warm woody base."
inputs = target_tokenizer(
    dummy_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)
inputs = {k: v.to(device) for k, v in inputs.items()}

start_t = time.time()
with torch.no_grad():
    for _ in range(50):
        _ = target_model(**inputs)

avg_ms = ((time.time() - start_t) / 50) * 1000

print("\n!!! COPY TO PAPER !!!")
print(f"Model Size     : {size_mb:.2f} MB")
print(f"Inference Time : {avg_ms:.2f} ms/sample")

target_model.to("cpu")

# ============================================================
# ABLATION STUDY: NOTES-ONLY (TF-IDF BASELINE)
# ============================================================

print("\n[Ablation] Notes-only baseline (TF-IDF)")

df_ablation = target_df[[name_col, notes_col]].dropna().copy()

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df_ablation[notes_col])
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

indices = pd.Series(df_ablation.index, index=df_ablation[name_col]).drop_duplicates()

test_perfume = (
    "Coco Mademoiselle"
    if "Coco Mademoiselle" in indices
    else indices.index[0]
)

idx = indices[test_perfume]
sim_scores = sorted(
    list(enumerate(cosine_sim[idx])),
    key=lambda x: x[1],
    reverse=True
)[1:6]

recs = df_ablation.iloc[[i[0] for i in sim_scores]][name_col].tolist()

print(f"\n!!! COPY TO PAPER (Baseline – {test_perfume}) !!!")
for i, r in enumerate(recs, 1):
    print(f"{i}. {r}")